# Topic 5: Production Streaming Patterns

> **Phase 6 → Week 11 → Topic 5**

---

## Production Streaming Checklist

```
Before shipping a streaming job to production:

✓ checkpointLocation on durable storage (S3/HDFS — NOT local disk)
✓ Watermark set on all stateful operations
✓ maxOffsetsPerTrigger set (prevent backlog OOM)
✓ Graceful shutdown: query.stop() on SIGTERM
✓ Schema is explicit (never inferSchema in streaming)
✓ Dead letter sink for malformed/corrupt records
✓ Monitoring: lastProgress logged every N triggers
✓ Alerting: on query failure (exception hook)
✓ Resource sizing: executor memory >= state store size
✓ Spark UI reviewed: no spill, no skew in state partitions
```

---

## Interview Cheat Sheet

**Q: How do you handle schema evolution in a streaming pipeline?**
> For file sources: update the explicit schema definition and restart the query. For Kafka JSON: use a permissive `from_json()` — new fields appear as columns, missing fields as null. Avoid `inferSchema`. For Avro/Protobuf: use a Schema Registry — Spark can fetch the latest schema automatically.

**Q: Your streaming job has been running for 3 days and suddenly slows down. How do you diagnose?**
> Check: (1) Spark UI Streaming tab — is `inputRowsPerSecond` increasing (backlog)? (2) State store size — growing unboundedly (missing watermark)? (3) Spill in the Stages tab — executor running out of memory? (4) Kafka consumer lag — are we falling behind the producers? (5) GC pauses in executor logs.

**Q: How do you restart a streaming job safely?**
> Stop the job gracefully (SIGTERM → `query.stop()`), verify the checkpoint is intact, then restart with the SAME checkpoint location. Spark will read the committed offsets and resume from where it left off. Never delete the checkpoint unless you want to reprocess from the beginning.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time, os, shutil, signal, threading, json
from datetime import datetime

spark = SparkSession.builder \
    .appName("Week11 - Production Streaming") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready")

## Part 1: Production Pipeline Template

In [ ]:
# Production streaming pipeline template
# Demonstrates: dead letter, monitoring, graceful shutdown

GOOD_OUT  = "/tmp/prod_good"
BAD_OUT   = "/tmp/prod_bad"
CKPT_GOOD = "/tmp/ckpt_prod_good"
CKPT_BAD  = "/tmp/ckpt_prod_bad"

for p in [GOOD_OUT, BAD_OUT, CKPT_GOOD, CKPT_BAD]:
    if os.path.exists(p): shutil.rmtree(p)

payload_schema = StructType([
    StructField("event_id",   StringType()),
    StructField("amount",     DoubleType()),
    StructField("category",   StringType()),
    StructField("customer_id",StringType()),
])

# Simulate incoming stream (mix of valid and invalid JSON)
raw = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("raw_value",
        F.when(
            F.col("value") % 10 == 0,  # 10% bad records
            F.lit("NOT_JSON_garbage_@#$")
        ).otherwise(
            F.to_json(F.struct(
                F.concat(F.lit("E"), F.col("value").cast("string")).alias("event_id"),
                ((F.col("value") % 999 + 1).cast("double")).alias("amount"),
                F.element_at(
                    F.array(F.lit("electronics"), F.lit("food"), F.lit("clothing")),
                    (F.col("value") % 3 + 1).cast("int")
                ).alias("category"),
                F.concat(F.lit("C"), (F.col("value") % 5 + 1).cast("string")).alias("customer_id"),
            ))
        )
    )

# Parse JSON: corrupt records result in null struct
parsed = raw.withColumn("data", F.from_json(F.col("raw_value"), payload_schema))

# Split: valid vs dead letter
valid   = parsed.filter(F.col("data").isNotNull()).select("timestamp", "data.*")
invalid = parsed.filter(F.col("data").isNull()) \
    .select("timestamp", F.col("raw_value").alias("bad_payload"),
            F.lit("JSON_PARSE_ERROR").alias("error_code"))

# Monitoring state
batch_stats = []

def write_valid(df, batch_id):
    count = df.count()
    df.write.mode("append").parquet(GOOD_OUT)
    batch_stats.append({"batch_id": batch_id, "valid": count})

def write_invalid(df, batch_id):
    count = df.count()
    if count > 0:
        df.write.mode("append").parquet(BAD_OUT)
    # Update monitoring
    if batch_stats and batch_stats[-1].get("batch_id") == batch_id:
        batch_stats[-1]["invalid"] = count
        pct = count / (count + batch_stats[-1].get("valid", 0)) * 100 if count > 0 else 0
        if pct > 15:
            print(f"  ⚠️  ALERT: batch {batch_id} rejection rate {pct:.1f}% > 15% threshold")

q_valid = valid.writeStream \
    .foreachBatch(write_valid) \
    .option("checkpointLocation", CKPT_GOOD) \
    .trigger(processingTime="3 seconds") \
    .start()

q_invalid = invalid.writeStream \
    .foreachBatch(write_invalid) \
    .option("checkpointLocation", CKPT_BAD) \
    .trigger(processingTime="3 seconds") \
    .start()

print("Production pipeline running (valid + dead letter streams)...")
time.sleep(15)

# Graceful shutdown
q_valid.stop()
q_invalid.stop()
print("\nBatch stats:")
for s in batch_stats:
    print(f"  Batch {s['batch_id']}: valid={s.get('valid',0)}, invalid={s.get('invalid',0)}")

good_total = spark.read.parquet(GOOD_OUT).count() if os.path.exists(GOOD_OUT) else 0
bad_total  = spark.read.parquet(BAD_OUT).count()  if os.path.exists(BAD_OUT)  else 0
print(f"\nTotal valid: {good_total}, Total dead letter: {bad_total}")

## Part 2: Monitoring & Progress Logging

In [ ]:
# Production monitoring: log progress every N triggers
def monitor_query(query, interval_s=5, max_iterations=4):
    """Log streaming query progress at regular intervals."""
    for i in range(max_iterations):
        time.sleep(interval_s)
        if not query.isActive:
            print("Query stopped.")
            break
        p = query.lastProgress
        if p:
            ts_str = datetime.now().strftime("%H:%M:%S")
            print(
                f"[{ts_str}] batch={p['batchId']} "
                f"in={p['numInputRows']} "
                f"rate_in={p['inputRowsPerSecond']:.0f}/s "
                f"rate_out={p['processedRowsPerSecond']:.0f}/s "
                f"trigger={p.get('triggerExecution', {}).get('triggerDuration', '?')}ms"
            )
            # Alert: input rate much higher than processing rate (falling behind)
            if (p['inputRowsPerSecond'] > 0 and
                p['processedRowsPerSecond'] < p['inputRowsPerSecond'] * 0.8):
                print(f"  ⚠️  ALERT: processing rate falling behind input rate!")

stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 100) \
    .load()

q_mon = stream \
    .withColumn("x", F.col("value") * 2) \
    .writeStream \
    .format("memory") \
    .queryName("monitor_demo") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

monitor_query(q_mon, interval_s=3, max_iterations=4)
q_mon.stop()

## Part 3: Schema Evolution in Streaming

In [ ]:
print("""
Schema Evolution Strategies:
════════════════════════════════════════════════════════════════

Problem: upstream adds a new field to JSON payloads.
  Old: {"order_id": "O1", "amount": 100}
  New: {"order_id": "O1", "amount": 100, "region": "south"}

Strategy 1: Add new field to schema (permissive)
  old_schema = StructType([StructField("order_id", ...), StructField("amount", ...)])
  new_schema = old_schema.add(StructField("region", StringType()))  # add nullable field
  
  → Old records: region = null
  → New records: region = "south"
  → No pipeline restart needed if schema is computed dynamically

Strategy 2: Select only known fields
  parsed = raw.withColumn("data", from_json("value", full_schema))
  safe = parsed.select("data.order_id", "data.amount")  # ignore unknown fields
  → Unknown fields in JSON are ignored (from_json is permissive by default)

Strategy 3: Schema Registry (Avro/Protobuf)
  For Kafka + Confluent Schema Registry:
  .option("schema.registry.url", "http://registry:8081")
  Schema evolution rules: BACKWARD, FORWARD, FULL compatibility
  → Schema versioned and centrally managed

Strategy 4: Raw string passthrough
  Don't parse at ingestion — store raw JSON as string.
  Parse at consumption time (Silver layer).
  → Most flexible but defers schema enforcement

Golden rule: make your schema ADDITIVE
  ✓ Adding nullable fields: safe
  ✗ Removing fields: breaks downstream
  ✗ Renaming fields: breaks downstream
  ✗ Changing types: breaks downstream
════════════════════════════════════════════════════════════════
""")

## Part 4: Production Deployment Script Template

In [ ]:
print(r"""
Production Streaming Job Template (streaming_job.py):
════════════════════════════════════════════════════════════════

import argparse, signal, sys, logging, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("streaming_job")

CONFIGS = {
    "dev": {
        "kafka_servers": "localhost:9092",
        "input_topic": "orders-dev",
        "output_path": "/tmp/streaming_output",
        "checkpoint": "/tmp/streaming_checkpoint",
        "trigger_interval": "10 seconds",
        "watermark": "1 minute",
        "max_offsets": 1000,
    },
    "prod": {
        "kafka_servers": "kafka-prod:9092",
        "input_topic": "orders",
        "output_path": "s3://my-bucket/streaming/orders",
        "checkpoint": "s3://my-bucket/checkpoints/orders",
        "trigger_interval": "30 seconds",
        "watermark": "15 minutes",
        "max_offsets": 100000,
    },
}

ORDER_SCHEMA = StructType([
    StructField("order_id",    StringType()),
    StructField("customer_id", StringType()),
    StructField("amount",      DoubleType()),
    StructField("category",    StringType()),
    StructField("event_time",  TimestampType()),
])

def build_spark(env):
    return SparkSession.builder \
        .appName(f"OrdersStreaming-{env}") \
        .config("spark.sql.shuffle.partitions",
                "4" if env == "dev" else "200") \
        .getOrCreate()

def run(env):
    cfg = CONFIGS[env]
    spark = build_spark(env)

    raw = spark.readStream.format("kafka") \
        .option("kafka.bootstrap.servers", cfg["kafka_servers"]) \
        .option("subscribe", cfg["input_topic"]) \
        .option("startingOffsets", "latest") \
        .option("maxOffsetsPerTrigger", cfg["max_offsets"]) \
        .load()

    parsed = raw \
        .withColumn("data", F.from_json(F.col("value").cast("string"), ORDER_SCHEMA)) \
        .withColumn("event_time", F.col("data.event_time")) \
        .withWatermark("event_time", cfg["watermark"])

    def process_batch(df, batch_id):
        valid   = df.filter(F.col("data").isNotNull())
        invalid = df.filter(F.col("data").isNull())
        valid.select("event_time", "data.*") \
             .write.mode("append").parquet(cfg["output_path"])
        log.info(f"Batch {batch_id}: {valid.count()} valid, {invalid.count()} invalid")

    query = parsed.writeStream \
        .foreachBatch(process_batch) \
        .option("checkpointLocation", cfg["checkpoint"]) \
        .trigger(processingTime=cfg["trigger_interval"]) \
        .start()

    def handle_sigterm(sig, frame):
        log.info("SIGTERM received — stopping query gracefully")
        query.stop()
        sys.exit(0)

    signal.signal(signal.SIGTERM, handle_sigterm)
    query.awaitTermination()  # blocks until stopped

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--env", default="dev", choices=["dev", "prod"])
    args = parser.parse_args()
    run(args.env)

# Run with:
# spark-submit \\
#   --master yarn --deploy-mode cluster \\
#   --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 \\
#   streaming_job.py --env prod
════════════════════════════════════════════════════════════════
""")

## Exercises

1. Take the production template above and add a monitoring thread that logs `lastProgress` every 30 seconds. What fields in `lastProgress` would you alert on in production?
2. Your streaming job is writing to Parquet. The upstream schema adds a new field `region`. Write the code to handle this gracefully without a pipeline restart. What option do you set?
3. Explain why checkpoints should be on S3/HDFS and not local disk in production. What happens if a Kubernetes pod running your streaming job is evicted and rescheduled on a different node?
4. You have two streaming queries (valid + dead letter) with separate checkpoints. What happens if the valid query stops but the dead letter query continues? What data integrity risk does this create?
5. Design a streaming pipeline for: Kafka → parse → validate (dead letter bad records) → enrich (join static catalog) → 10-minute tumbling window count → write to Delta. Draw the DAG and identify which stages are stateful.